<a href="https://colab.research.google.com/github/ulugbek-lab/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Menu detector!")

Menu detector!


In [2]:
# ------------------------------
# Import Libraries
# ------------------------------
from google.colab import drive
import torchvision.transforms as transforms


from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print(DATASET_PATH)



CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot dog': 'hot-dog',
    'chocolate_cake': 'dessert',  # label grouping | class consolidation(such as dessert)
    'cheesecake': 'dessert',      # label grouping | class consolidation(such as dessert)
    'kebab': 'kebab',
    'sushi': 'sushi'
}




CLASSES = ['hamburger', 'hot-dog', 'dessert', 'kebab', 'sushi']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)} #assign number to classes
NUM_CLASSES = len(CLASSES)
print(CLASS_TO_IDX)
print(NUM_CLASSES) # 5 categories/classes


transform = transforms.Compose([
    transforms.Resize((224,224)), # resize image width & hight
    transforms.ToTensor(), # convert image to tensor; # H, W, C  => C, H, W(tensor transform)
    transforms.Normalize(mean=[0.485, 0.456,0.406],# pixel = (pixel - mean) /std
                         std=[0.229, 0.224, 0.225])
])




/content/drive/MyDrive/food101_dataset
{'hamburger': 0, 'hot-dog': 1, 'dessert': 2, 'kebab': 3, 'sushi': 4}
5


In [5]:


# ------------------------------
# Custom Dataset class
# ------------------------------



class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print('images_length', len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    print('img_path',img_path)
    label = self.labels[idx]
    print('label',label)

    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((idx + 1) % len(self.images))# skip error image path
    if self.transform:
      image = self.transform(image)

    return image, label
#


In [11]:
# ------------------------------
# Gather and Split class
# ------------------------------

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
   class_path = os.path.join(DATASET_PATH, original_class)
   print('Class_path:', class_path)
   if not os.path.exists(class_path):
     print(f"Warning: {class_path} not found")
     continue
   for img in os.listdir(class_path):
     if img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images) # mix all the images
split = int(0.8 * len(all_images))
train_data = all_images[:split] # 1000 | 800 train_data | 200 val_data
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)


# print('all_images', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[7]






Class_path: /content/drive/MyDrive/food101_dataset/hamburger
Class_path: /content/drive/MyDrive/food101_dataset/hot dog
Class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
Class_path: /content/drive/MyDrive/food101_dataset/cheesecake
Class_path: /content/drive/MyDrive/food101_dataset/kebab
Class_path: /content/drive/MyDrive/food101_dataset/sushi
images_length 1920
1920
/content/drive/MyDrive/food101_dataset/hamburger/3912017.jpg
0
